In [ ]:
# ======================================================
# PHASE 3 - STEP 9 : LABEL ENCODING
# ======================================================

LABEL2ID = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

ID2LABEL = {
    v: k
    for k, v in LABEL2ID.items()
}

train_df["label"] = train_df["answer"].map(LABEL2ID)

print(train_df[["answer", "label"]].head())

print("\nUnique labels:")
print(sorted(train_df["label"].unique()))

In [ ]:
# ======================================================
# PHASE 3 - STEP 10 : MULTIPLE CHOICE DATASET
# ======================================================

mcq_train = pd.DataFrame({
    "id": train_df["id"],
    "prompt": train_df["prompt"],
    "choices": train_df[["A", "B", "C", "D", "E"]].values.tolist(),
    "label": train_df["label"]
})

mcq_test = pd.DataFrame({
    "id": test_df["id"],
    "prompt": test_df["prompt"],
    "choices": test_df[["A", "B", "C", "D", "E"]].values.tolist()
})

print(mcq_train.shape)
display(mcq_train.head())

print("\n")

print(mcq_test.shape)
display(mcq_test.head())

In [ ]:
# ======================================================
# PHASE 4 - STEP 11 : INSTALL LIBRARIES
# ======================================================

!pip install -q transformers datasets accelerate sentencepiece evaluate

In [ ]:
# ======================================================
# PHASE 4 - STEP 12 : IMPORTS
# ======================================================

import torch
import numpy as np

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
    DataCollatorForMultipleChoice
)

print(torch.__version__)
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# ======================================================
# PHASE 4 - STEP 13 : BUILD MCQ DATASET
# ======================================================

from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["prompt", "A", "B", "C", "D", "E", "label"]]
)

test_dataset = Dataset.from_pandas(
    test_df[["prompt", "A", "B", "C", "D", "E"]]
)

print(train_dataset)
print(test_dataset)

In [ ]:
# ======================================================
# PHASE 4 - STEP 14 : LOAD TOKENIZER
# ======================================================

from transformers import AutoTokenizer

MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loaded:", MODEL_NAME)

In [ ]:
# ======================================================
# PHASE 4 - STEP 15 : TOKENIZATION
# ======================================================

MAX_LENGTH = 256

ending_names = ["A", "B", "C", "D", "E"]

def preprocess_function(examples):

    first_sentences = [
        [prompt] * 5
        for prompt in examples["prompt"]
    ]

    second_sentences = [
        [
            examples["A"][i],
            examples["B"][i],
            examples["C"][i],
            examples["D"][i],
            examples["E"][i],
        ]
        for i in range(len(examples["prompt"]))
    ]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    return {
        k: [
            tokenized[k][i:i+5]
            for i in range(0, len(tokenized[k]), 5)
        ]
        for k in tokenized
    }

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True
)

print(tokenized_train)